# Phase 4 - Stage 1 (Kaggle or Colab)

Steps 4.3 and 4.4 of `PLAN.md`: runs 3-8 of the experiment matrix.

| Run | Model | Varies | Step |
|---|---|---|---|
| 3-6 | BiLSTM + attention | embedding matrix E0/E1/E2/E3 | 4.3 |
| 7 | `bert-base-uncased` | - | 4.4 |
| 8 | BiomedBERT | - | 4.4 |

Runs 1-2 (Naive Bayes, TF-IDF LogReg/SVM) already ran locally in seconds;
their rows are in `results/runs.csv` and their table is in
`results/figures/stage1_baselines.md`.

**Accelerator: GPU T4 x2** (cell 1 pins the job to a single card - see there
for why). **Internet: On**, so runs 7-8 can pull the checkpoints from the
HF Hub; attach them as a Dataset instead if internet is unavailable.

### The one thing that can invalidate this notebook

Runs 3-6 are a single-variable ablation. If one of them sees a different
GPU count than another, its effective batch size differs too, and the
four-bar chart no longer compares embeddings - it compares batch sizes
(`PLAN.md` F8). Cell 1 pins `CUDA_VISIBLE_DEVICES=0` for the whole session
and every run row records `device_count`. It has to be the FIRST cell you run,
and if you restart the kernel mid-ablation you must run it first again.


## 1. Environment, and the single-GPU pin (PLAN F8)

**This is the first cell for a reason.** `CUDA_VISIBLE_DEVICES` is read once,
when the CUDA context initialises. Set it after anything has already touched
`torch.cuda` - even a `device_count()` call in a print statement - and it is
silently ignored for the rest of the kernel's life. So the pin goes above the
`import torch`, not in a later cell.

### Why one GPU

`nn.DataParallel` multiplies the per-device batch by the number of visible
devices. Two GPUs would train runs 3-6 at an effective batch of 64 where one
trains at 32 - and if that differed *between* the four runs, the ablation
would be measuring batch size rather than embeddings.

One GPU is the right setting for both stages regardless: the BiLSTM is small
enough that the second card buys nothing, and `pytorch-crf` in Phase 5 does
not survive `DataParallel` cleanly.

### Which accelerator to select

| Kaggle setting | Result | Verdict |
|---|---|---|
| **GPU T4 x2** | this cell pins to one T4; the second card idles | **use this** |
| GPU P100 | one card, no pin needed | works, but slower for runs 7-8 |

Both give a single GPU. Prefer T4: it has tensor cores, so the `fp16=True`
in the BERT recipe actually buys throughput, where the P100 has to emulate
half precision. Leaving one T4 idle costs nothing - Kaggle bills the session,
not the card.


In [ ]:
import os

# BEFORE importing torch. See the note above - this is not stylistic.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import sys, subprocess, pathlib, time

ON_KAGGLE = pathlib.Path('/kaggle').exists()
ON_COLAB = 'google.colab' in sys.modules or pathlib.Path('/content').exists()
PLATFORM = 'kaggle' if ON_KAGGLE else 'colab' if ON_COLAB else 'local'
WORK = pathlib.Path('/kaggle/working' if ON_KAGGLE else
                    '/content' if ON_COLAB else '.')
print('platform :', PLATFORM)
print('workdir  :', WORK)

import torch, transformers, datasets, numpy, sklearn
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| datasets', datasets.__version__)
print('numpy', numpy.__version__, '| sklearn', sklearn.__version__)

n = torch.cuda.device_count()
print('CUDA devices visible:', n)
print('GPU:', torch.cuda.get_device_name(0) if n else 'CPU')
assert n <= 1, (
    f'{n} GPUs are visible, so the pin did not take effect. Something touched '
    'torch.cuda before this cell ran. Fix: Run -> Restart & clear cell outputs, '
    'then run THIS cell first (PLAN F8).')

# GROUP B of requirements-remote.txt: record these, do not reinstall them.
# Paste this line into the notes column if a run row ever needs auditing.
print(f'\nRECORD: torch={torch.__version__} transformers={transformers.__version__} '
      f'datasets={datasets.__version__} numpy={numpy.__version__} devices={n}')


## 2. Get the code

The frozen splits are committed (`PLAN.md` F7), so a clone brings the task
corpus with it and nothing has to be re-split here.


In [ ]:
REPO_URL = 'https://github.com/sifatul-islam-onik/ADE-Sentinel.git'
REPO = WORK / 'ADE-Sentinel'

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=False)

sys.path.insert(0, str(REPO))
os.chdir(REPO)
GIT_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('repo at', GIT_COMMIT)

for split in ('train', 'dev', 'test'):
    import pandas as pd
    df = pd.read_parquet(REPO / 'data' / 'splits' / f'stage1_{split}.parquet')
    print(f'  stage1_{split}: {len(df):,} rows, {df.label.mean():.1%} positive')


## 3. Test before training

The BiLSTM tests are skipped locally because the authoring environment has
no torch (`PLAN.md` F7). This is the first machine that can run them, so it
runs them - before spending GPU quota on four runs built on an attention
mask that leaks padding.

`test_padding_does_not_change_the_output` is the one that matters: that
failure mode produces plausible-looking numbers rather than an error.


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                'tests/test_stage1_models.py', 'tests/test_encoding.py',
                'tests/test_effective_batch.py'], check=True)


## 4. Embedding matrices

The four matrices are ~57 MB and are **gitignored on purpose** (`PLAN.md` F7),
so cloning the repo does not bring them. Only `vocab.json` arrives that way -
which is why the cell below insists on finding both together rather than
trusting a directory that merely has a vocabulary in it.

### One-time setup: upload them as a Kaggle Dataset

1. **Kaggle -> Datasets -> New Dataset**
2. Drag in all five files from your local `models/emb_matrices/`:
   `E0_random.npy`, `E1.npy`, `E2.npy`, `E3.npy`, `vocab.json`
3. Title it `ade-sentinel-artifacts`, visibility **Private**, Create
4. Back in this notebook: **Add Input** -> your dataset
5. Read the version off the Input panel and put it in `DATASET_VERSION` below

The cell finds the files whether they sit in the Dataset root (what dragging
individual files gives you) or inside an `emb_matrices/` folder.

**Record the version number.** It becomes the `dataset_version` column of every
run row, and it is what lets a number be traced back to the inputs that
produced it (`PLAN.md` F10). The matrices are not fixed forever - retrain
Word2Vec and E2 changes underneath you.


In [ ]:
DATASET_VERSION = ''   # <- fill in, e.g. 'ade-sentinel-artifacts v1'

import json, numpy as np


def holds_matrices(d):
    """A usable directory has the vocabulary AND the matrices it indexes.

    Both, not either. `models/emb_matrices/vocab.json` is committed while the
    .npy files are gitignored (PLAN F7), so the repo directory satisfies a
    vocab-only test and then fails later, inside training, with a missing-file
    error that points at the wrong thing.
    """
    return d.is_dir() and (d / 'vocab.json').exists() and any(d.glob('E*.npy'))


# Find the vocabulary wherever it landed and take its directory. Kaggle
# mounts a Dataset at /kaggle/input/<slug>, but files dragged in singly sit
# at the root while an uploaded folder or zip keeps its own level - and the
# path shown in the sidebar is not always the mount path. Searching for the
# file instead of guessing the layout makes all of that irrelevant.
candidates = []
if ON_KAGGLE:
    root = pathlib.Path('/kaggle/input')
    candidates += [p.parent for p in root.rglob('vocab.json')]
    candidates += [root] + sorted(root.glob('*')) + sorted(root.glob('*/*'))
if ON_COLAB:
    candidates += [pathlib.Path('/content/drive/MyDrive/ade-sentinel/emb_matrices')]
candidates.append(REPO / 'models' / 'emb_matrices')

MATRICES = next((d for d in candidates if holds_matrices(d)), None)

if MATRICES is None:
    # Show what is actually mounted, not just which candidates were dirs -
    # 'searched N directories' tells you nothing when the problem is that
    # the dataset is attached under a name you did not expect.
    listing = []
    if ON_KAGGLE and pathlib.Path('/kaggle/input').exists():
        for p in sorted(pathlib.Path('/kaggle/input').rglob('*'))[:40]:
            listing.append(f'    {p}{"/" if p.is_dir() else ""}')
    raise SystemExit(
        'No directory with both vocab.json and E*.npy was found.\n\n'
        'Actually mounted under /kaggle/input:\n'
        + ('\n'.join(listing) if listing else '    (nothing - no Dataset attached)')
        + '\n\nFix: Kaggle -> Datasets -> New Dataset, upload ALL FIVE files '
        'from models/emb_matrices/ (E0_random.npy, E1.npy, E2.npy, E3.npy, '
        'vocab.json), then Add Input on this notebook and re-run this cell.\n'
        'The .npy files are gitignored (PLAN F7), so cloning the repo does '
        'NOT bring them - only vocab.json arrives that way.')

vocab = json.load(open(MATRICES / 'vocab.json'))
found = sorted(MATRICES.glob('E*.npy'))
print('matrices :', MATRICES)
print('vocab    :', f'{len(vocab):,} rows')
for f in found:
    rows, dim = np.load(f, mmap_mode='r').shape
    flag = '' if rows == len(vocab) else '   <-- ROW COUNT MISMATCH'
    print(f'  {f.name:16s} ({rows:,}, {dim}){flag}')

missing = {'E0_random', 'E1', 'E2', 'E3'} - {f.stem for f in found}
assert not missing, (
    f'missing {sorted(missing)} - runs 3-6 need all four, and an ablation '
    'with a bar absent is not an ablation')
assert all(np.load(f, mmap_mode='r').shape[0] == len(vocab) for f in found), (
    'a matrix disagrees with vocab.json on row count, so the two came from '
    'different builds. Re-run scripts/build_embedding_matrices.py and '
    're-upload all five files together.')

if not DATASET_VERSION:
    print('\nWARNING: DATASET_VERSION is blank - fill it in before training, '
          'or the run rows lose their input provenance (PLAN F10)')


## 5. Runs 3-6 - the embedding ablation

**This is the project's headline experiment** (PRD section 7). One command
per run; `--embedding` is the only flag that differs between them.

| Run | Matrix | What it is |
|---|---|---|
| 3 | `E0_random` | random init - the floor |
| 4 | `E1` | GloVe 300d, general purpose |
| 5 | `E2` | our Word2Vec, PubMed |
| 6 | `E3` | our FastText, PubMed |

### Frozen embeddings

These four freeze the embedding layer, which is what makes the run a
measurement *of the vectors*. Let the layer train and E0's random rows learn
task-specific values from 14.6k sentences, so the floor lifts and the
comparison partly becomes one about optimisation rather than pretraining.

Section 7 then repeats all four fine-tuned. Both conditions are reported:
frozen answers *how good are these vectors*, fine-tuned answers *does the
initialisation still matter once the task has had its say*. Reporting only
the flattering one is choosing the answer in advance, and the PRD is explicit
that a well-explained negative result beats a suspicious positive one.

Expect a few minutes per run on one T4.


In [ ]:
def train_bilstm(run_id, embedding, unfreeze=False, **kw):
    cmd = [sys.executable, 'scripts/train_bilstm.py',
           '--run-id', str(run_id), '--embedding', embedding,
           '--matrices', str(MATRICES), '--dataset-version', DATASET_VERSION]
    if unfreeze:
        cmd.append('--unfreeze-embeddings')
    for k, v in kw.items():
        cmd += [f'--{k.replace("_", "-")}', str(v)]
    print('\n' + '=' * 70)
    print(' '.join(cmd[1:]))
    print('=' * 70)
    subprocess.run(cmd, check=True)


ABLATION = [('3', 'E0_random'), ('4', 'E1'), ('5', 'E2'), ('6', 'E3')]

t0 = time.time()
for run_id, embedding in ABLATION:
    train_bilstm(run_id, embedding)
print(f'\nruns 3-6 complete in {(time.time() - t0) / 60:.1f} min')


## 6. Runs 3u-6u - the same ablation, embeddings fine-tuned

The corroborating condition. If the ordering E0 < E1 < E2 <= E3 survives
fine-tuning, the claim is robust. If the gaps close, that is the more
interesting finding and it belongs in the report: it would say the domain
advantage is an *initialisation* advantage that task supervision can partly
recover, which is a real and defensible conclusion.

Skip this section if GPU quota is short - runs 3-6 are the required ones.


In [ ]:
for run_id, embedding in ABLATION:
    train_bilstm(f'{run_id}u', embedding, unfreeze=True)


## 7. Runs 7-8 - the transformers

The same domain-vs-general question at the contextual level: identical
architecture, different pretraining corpus. PRD 8.1 wants that symmetry with
runs 4-6.

The PRD recipe is **effective batch 16**, lr 2e-5, 3 epochs, max_len 128,
fp16. `scripts/train_bert.py` takes the *effective* batch and derives the
per-device value from the visible GPU count, so the recipe survives whichever
machine this lands on (F8). With cell 1's pin that is 16 on one device.

Roughly 10-20 minutes each on a T4. Needs Internet on for the HF download.


In [ ]:
def train_bert(run_id, model, **kw):
    cmd = [sys.executable, 'scripts/train_bert.py',
           '--run-id', str(run_id), '--model', model,
           '--dataset-version', DATASET_VERSION]
    for k, v in kw.items():
        cmd += [f'--{k.replace("_", "-")}', str(v)]
    print('\n' + '=' * 70)
    print(' '.join(cmd[1:]))
    print('=' * 70)
    subprocess.run(cmd, check=True)


t0 = time.time()
train_bert('7', 'bert-base-uncased')
train_bert('8', 'biomedbert')
print(f'\nruns 7-8 complete in {(time.time() - t0) / 60:.1f} min')


## 8. The results table and the four-bar chart

Step 4.5. Reads `results/runs.csv` - so it covers runs 1-2 from the local
session as well - and writes the figures the report needs.


In [ ]:
subprocess.run([sys.executable, 'scripts/stage1_report.py'], check=True)

print(open('results/figures/stage1_results.md', encoding='utf-8').read())


## 9. Save the outputs - before the session ends

Kaggle discards `/kaggle/working` when the notebook stops; Colab loses it
with the VM. Nothing here is recoverable afterwards.

| Artefact | Where it goes |
|---|---|
| `runs.csv` rows | **commit to git** - this is the experiment record |
| `stage1_results.md`, `stage1_embeddings.png` | **commit to git** - report figures |
| checkpoints, `test_predictions.npz` | Kaggle Dataset - Phase 6 needs the predictions |

The predictions matter more than they look: Phase 6.4 samples 30 pipeline
failures for the error taxonomy, and re-running a GPU job to recover numbers
you already computed is quota spent on nothing.


In [ ]:
import shutil

bundle = WORK / 'phase4_outputs'
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()

shutil.copy2('results/runs.csv', bundle / 'runs.csv')
for fig in pathlib.Path('results/figures').glob('stage1_*'):
    shutil.copy2(fig, bundle / fig.name)

if pathlib.Path('models/stage1').exists():
    shutil.copytree('models/stage1', bundle / 'stage1_models',
                    ignore=shutil.ignore_patterns('trainer'))

total = sum(f.stat().st_size for f in bundle.rglob('*') if f.is_file())
print(f'{bundle}  ({total / 1e6:,.0f} MB)')
for f in sorted(bundle.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(bundle)}  {f.stat().st_size / 1e6:,.1f} MB')


### Commit the record back

`runs.csv` and the figures are small and belong in git. Download them from
the output panel and commit locally, or push from here with a token in
**Add-ons -> Secrets** (never paste one into a cell - Kaggle notebooks are
shareable and the output is saved with them).


---

## Phase 4 exit criterion

From `PLAN.md`: runs 1-8 in `runs.csv` with effective batch and device count
logged, the four-bar chart generated, and the best checkpoint saved.

Before moving to Phase 5, read the chart honestly:

- **E2/E3 above E1** corroborates the headline claim with the third and final
  evidence type. Coverage and neighbours already carried it (Phase 3); this
  makes it three for three.
- **E1 above E2/E3** is a real result, not a failure. Diagnose it - corpus
  size, `min_count`, frozen vs fine-tuned - and write it up. PRD section 12
  asks for exactly this.
- **Any BiLSTM below run 2's TF-IDF LogReg** means the BiLSTM is
  undertrained, not that embeddings do not help. Fix that before drawing any
  conclusion from the four bars.

Then carry the best embedding of E1-E3 into runs 9-10 (`PLAN.md` 5.4-5.5).
The BIO converter and its tests are already done.
